# Heart Disease Merged Dataset — Data Processing

**Input:**  `../raw/heart_disease_merged.csv` (Cleveland + Hungary + Switzerland + VA Long Beach, 920 rows)
**Output:** `../processed/heart_disease_processed.csv`

This notebook's only job is cleaning the merged raw data: **duplicates**, **illogical/invalid values**,
and **missing values**. No feature engineering, scaling, or modeling happens here — this is the
raw → processed step, and every downstream notebook should read from `data/processed/`, not `data/raw/`.

Pipeline:
1. Load & inspect
2. Remove exact duplicate rows
3. Detect and correct illogical values (convert to missing, since a wrong value is worse than a missing one)
4. Handle missing values (drop columns that are mostly empty, impute the rest)
5. Final validation + save


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

# Paths are relative to this notebook's location: BinX_Week_05/data/processing/
RAW_PATH = Path("../raw/heart_disease_merged.csv")
PROCESSED_DIR = Path("../processed")
PROCESSED_PATH = PROCESSED_DIR / "heart_disease_processed.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH.resolve()


WindowsPath('C:/Users/HP/Desktop/BinX_ML_Internship/BinX_Week_05/data/raw/heart_disease_merged.csv')

## 1. Load & inspect the merged raw data

In [2]:
df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)
df.head()


Shape: (920, 16)


,country,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,target
0,Cleveland (USA),63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0
1,Cleveland (USA),67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,1
2,Cleveland (USA),67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,1
3,Cleveland (USA),37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0
4,Cleveland (USA),41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   country   920 non-null    str    
 1   age       920 non-null    float64
 2   sex       920 non-null    float64
 3   cp        920 non-null    float64
 4   trestbps  861 non-null    float64
 5   chol      890 non-null    float64
 6   fbs       830 non-null    float64
 7   restecg   918 non-null    float64
 8   thalach   865 non-null    float64
 9   exang     865 non-null    float64
 10  oldpeak   858 non-null    float64
 11  slope     611 non-null    float64
 12  ca        309 non-null    float64
 13  thal      434 non-null    float64
 14  num       920 non-null    int64  
 15  target    920 non-null    int64  
dtypes: float64(13), int64(2), str(1)
memory usage: 115.1 KB


In [4]:
df.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
country,920,4,Cleveland (USA),303,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,920.0,NaN,NaN,NaN,53.51087,9.424685,28.0,47.0,54.0,60.0,77.0
sex,920.0,NaN,NaN,NaN,0.78913,0.408148,0.0,1.0,1.0,1.0,1.0
cp,920.0,NaN,NaN,NaN,3.25,0.930969,1.0,3.0,4.0,4.0,4.0
trestbps,861.0,NaN,NaN,NaN,132.132404,19.06607,0.0,120.0,130.0,140.0,200.0
chol,890.0,NaN,NaN,NaN,199.130337,110.78081,0.0,175.0,223.0,268.0,603.0
fbs,830.0,NaN,NaN,NaN,0.166265,0.372543,0.0,0.0,0.0,0.0,1.0
restecg,918.0,NaN,NaN,NaN,0.604575,0.805827,0.0,0.0,0.0,1.0,2.0
thalach,865.0,NaN,NaN,NaN,137.545665,25.926276,60.0,120.0,140.0,157.0,202.0
exang,865.0,NaN,NaN,NaN,0.389595,0.487941,0.0,0.0,0.0,1.0,1.0


In [5]:
df["country"].value_counts()


country
Cleveland (USA)        303
Hungary                294
VA Long Beach (USA)    200
Switzerland            123
Name: count, dtype: int64

## 2. Duplicate rows

Check for exact full-row duplicates (same values across every column, including `country`).
A duplicate here almost certainly means the same patient record was captured twice during the merge.


In [6]:
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}")
df[df.duplicated(keep=False)].sort_values(list(df.columns))


Exact duplicate rows: 2


,country,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,target
404,Hungary,49.0,0.0,2.0,110.0,NaN,0.0,0.0,160.0,0.0,0.0,NaN,NaN,NaN,0,0
405,Hungary,49.0,0.0,2.0,110.0,NaN,0.0,0.0,160.0,0.0,0.0,NaN,NaN,NaN,0,0
859,VA Long Beach (USA),58.0,1.0,3.0,150.0,219.0,0.0,1.0,118.0,1.0,0.0,NaN,NaN,NaN,2,1
907,VA Long Beach (USA),58.0,1.0,3.0,150.0,219.0,0.0,1.0,118.0,1.0,0.0,NaN,NaN,NaN,2,1


In [7]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print(f"Dropped {before - after} duplicate row(s). New shape: {df.shape}")


Dropped 2 duplicate row(s). New shape: (918, 16)


## 3. Illogical / clinically invalid values

These are values that parsed fine as numbers but are impossible or invalid given what the column
represents. Rather than guessing a replacement, each one is converted to `NaN` here and handled
properly in the missing-value step below — a silently wrong value is more dangerous than a
correctly-flagged missing one.

Checks performed:
- **`age`**: must be in a plausible human range (0, 120]
- **`trestbps`** (resting blood pressure): a living patient cannot have 0 resting blood pressure
- **`chol`** (serum cholesterol): a living patient cannot have 0 cholesterol — this is a known data
  artifact in the Switzerland/VA sites, where cholesterol simply wasn't recorded and was entered as 0
  instead of left blank
- **`oldpeak`** (ST depression induced by exercise): should be ≥ 0 by clinical definition; negative
  values are data-entry errors
- **Categorical/code columns** (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal`, `num`):
  must fall inside their documented set of valid codes


In [8]:
valid_codes = {
    "sex":     {0, 1},
    "cp":      {1, 2, 3, 4},
    "fbs":     {0, 1},
    "restecg": {0, 1, 2},
    "exang":   {0, 1},
    "slope":   {1, 2, 3},
    "ca":      {0, 1, 2, 3},
    "thal":    {3, 6, 7},
    "num":     {0, 1, 2, 3, 4},
}

issues = {}

# age out of plausible human range
mask = ~df["age"].between(0, 120, inclusive="right")
issues["age (out of range)"] = int(mask.sum())
df.loc[mask, "age"] = np.nan

# trestbps: resting blood pressure of 0 is physiologically impossible
mask = df["trestbps"] == 0
issues["trestbps == 0"] = int(mask.sum())
df.loc[mask, "trestbps"] = np.nan

# chol: cholesterol of 0 is physiologically impossible -> treat as not-recorded
mask = df["chol"] == 0
issues["chol == 0"] = int(mask.sum())
df.loc[mask, "chol"] = np.nan

# oldpeak: negative ST depression is a data-entry error
mask = df["oldpeak"] < 0
issues["oldpeak < 0"] = int(mask.sum())
df.loc[mask, "oldpeak"] = np.nan

# categorical/code columns outside their documented valid set
for col, valid in valid_codes.items():
    mask = ~df[col].isin(valid) & df[col].notna()
    issues[f"{col} (invalid code)"] = int(mask.sum())
    df.loc[mask, col] = np.nan

pd.Series(issues, name="rows_flagged_as_illogical").to_frame()


,rows_flagged_as_illogical
age (out of range),0
trestbps == 0,1
chol == 0,172
oldpeak < 0,12
sex (invalid code),0
cp (invalid code),0
fbs (invalid code),0
restecg (invalid code),0
exang (invalid code),0
slope (invalid code),0


## 4. Missing values

Missingness is recomputed *after* the illogical-value pass above, since some illogical values
(e.g. `chol == 0`) were converted to `NaN` and need to be counted here too.

**Strategy:**
- Columns missing in **more than 40%** of rows are dropped outright — at that level of missingness,
  imputed values would dominate the column and add noise rather than signal.
- Columns below that threshold are imputed:
  - Numeric columns → **median**, computed **per country** (falls back to the overall median if a
    country has no observed values for that column), since normal ranges/measurement practices
    differ by site.
  - Categorical/code columns → **mode**, computed the same way.


In [9]:
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct.to_frame("missing_%")


,missing_%
ca,66.3
thal,52.7
slope,33.4
chol,21.9
fbs,9.8
oldpeak,8.1
trestbps,6.5
thalach,6.0
exang,6.0
restecg,0.2


In [10]:
DROP_THRESHOLD = 40.0  # percent

cols_to_drop = missing_pct[missing_pct > DROP_THRESHOLD].index.tolist()
print("Dropping columns (>", DROP_THRESHOLD, "% missing ):", cols_to_drop)

df = df.drop(columns=cols_to_drop)
df.shape


Dropping columns (> 40.0 % missing ): ['ca', 'thal']


(918, 14)

In [11]:
numeric_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_cols = [c for c in ["sex", "cp", "fbs", "restecg", "exang", "slope"] if c in df.columns]

# Numeric columns: impute with the per-country median
for col in numeric_cols:
    df[col] = df.groupby("country")[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median())  # safety net if a country has zero observed values

# Categorical columns: impute with the per-country mode
def group_mode(s):
    m = s.mode()
    return m.iloc[0] if not m.empty else np.nan

for col in categorical_cols:
    df[col] = df.groupby("country")[col].transform(lambda s: s.fillna(group_mode(s)))
    overall_mode = group_mode(df[col])
    df[col] = df[col].fillna(overall_mode)

df.isna().sum().to_frame("missing_after_imputation")


,missing_after_imputation
country,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0


## 5. Final validation & type cleanup

Cast the code/categorical columns and integer-valued fields to `int` now that they're fully
imputed, and do one last pass to confirm the processed data is clean.


In [12]:
int_like_cols = [c for c in ["sex", "cp", "fbs", "restecg", "exang", "slope", "num", "target"] if c in df.columns]
for col in int_like_cols:
    df[col] = df[col].astype(int)

assert df.isna().sum().sum() == 0, "There are still missing values left in the dataset."
assert df.duplicated().sum() == 0, "There are still duplicate rows left in the dataset."

print("Final shape:", df.shape)
df.dtypes


Final shape: (918, 14)


country         str
age         float64
sex           int64
cp            int64
trestbps    float64
chol        float64
fbs           int64
restecg       int64
thalach     float64
exang         int64
oldpeak     float64
slope         int64
num           int64
target        int64
dtype: object

In [13]:
df.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
country,918,4,Cleveland (USA),303,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,918.0,NaN,NaN,NaN,53.510893,9.432617,28.0,47.0,54.0,60.0,77.0
sex,918.0,NaN,NaN,NaN,0.78976,0.407701,0.0,1.0,1.0,1.0,1.0
cp,918.0,NaN,NaN,NaN,3.251634,0.931031,1.0,3.0,4.0,4.0,4.0
trestbps,918.0,NaN,NaN,NaN,132.130719,17.927526,80.0,120.0,130.0,140.0,200.0
chol,918.0,NaN,NaN,NaN,244.171024,52.03979,85.0,217.25,236.0,267.0,603.0
fbs,918.0,NaN,NaN,NaN,0.150327,0.357586,0.0,0.0,0.0,0.0,1.0
restecg,918.0,NaN,NaN,NaN,0.603486,0.805968,0.0,0.0,0.0,1.0,2.0
thalach,918.0,NaN,NaN,NaN,136.514161,25.483104,60.0,120.0,138.0,155.75,202.0
exang,918.0,NaN,NaN,NaN,0.423747,0.494421,0.0,0.0,0.0,1.0,1.0


## 6. Save processed data

In [14]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved processed dataset to: {PROCESSED_PATH.resolve()}")
print(f"Final shape: {df.shape}")


Saved processed dataset to: C:\Users\HP\Desktop\BinX_ML_Internship\BinX_Week_05\data\processed\heart_disease_processed.csv
Final shape: (918, 14)


## Summary

| Step | Result |
|---|---|
| Duplicates removed | rows dropped where every column matched exactly |
| Illogical values corrected | `age` out of range, `trestbps == 0`, `chol == 0`, `oldpeak < 0`, and any out-of-range categorical codes were converted to `NaN` |
| High-missingness columns dropped | any column missing in more than 40% of rows |
| Remaining missing values | imputed per-country (median for numeric, mode for categorical) |
| Output | `data/processed/heart_disease_processed.csv`, fully clean — no missing values, no duplicates |

Downstream notebooks (EDA, feature engineering, modeling) should read from `data/processed/`, never from `data/raw/`.
